# Resnet100

In [8]:
!pip3 install insightface onnxruntime opencv-python
import cv2, torch, insightface, os
import numpy as np
from insightface.app import FaceAnalysis
from insightface.data import get_image as ins_get_image
from torch.nn.functional import cosine_similarity
print("Current working directory:", os.getcwd())

# Load ResNet100
app = FaceAnalysis(providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
app.prepare(ctx_id=-1, det_size=(640, 640))

Current working directory: g:\.thesis\named-ai\data-preprocessing\Face_Dataset\MobileFaceNet


c:\Users\julia\AppData\Local\Programs\Python\Python311\Lib\site-packages\onnxruntime\capi\onnxruntime_inference_collection.py:121: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\julia/.insightface\models\buffalo_l\1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\julia/.insightface\models\buffalo_l\2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\julia/.insightface\models\buffalo_l\det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\julia/.insightface\models\buffalo_l\genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\julia/.insightface\models\buffalo_l\w600k_r50.onnx recognition ['None', 3, 112, 112] 127.

In [16]:
#get embeddings
def get_embeddings_resnet100(image_path):
    img = cv2.imread(image_path)
    if img is None:
        raise FileNotFoundError(f"Image not found or unreadable: {image_path}")

    faces = app.get(img)
    if len(faces) == 0:
        print(f"No face detected in {image_path}")
        return None
    print(f"Face detected in {image_path}")
    return torch.tensor(faces[0].embedding).squeeze()

In [17]:
#recognize face
def recognize_face(test_embedding, face_db, threshold=0.6):
    if test_embedding is None:
        return "Unknown", 0.0

    max_sim = 0
    identity = "Unknown"

    for name, db_embedding in face_db.items():
        sim = cosine_similarity(test_embedding.unsqueeze(0), db_embedding.unsqueeze(0))
        sim_val = sim.item()

        if sim_val > max_sim and sim_val > threshold:
            max_sim = sim_val
            identity = name

    return identity, max_sim

In [ ]:
print("Current working directory:", os.getcwd())

# build db
face_db_r100 = {}
face_db_r100["Kyle"] = get_embeddings_resnet100("my_images/kyle_183.jpg")
face_db_r100["Tom Cruise"] = get_embeddings_resnet100("my_images/Tom Cruise_12.jpg")
face_db_r100["Tom CruiseTest"] = get_embeddings_resnet100("my_images/testtom.jpg")

Current working directory: g:\.thesis\named-ai\data-preprocessing\Face_Dataset\MobileFaceNet
Face detected in my_images/testtom.jpg


In [28]:
#test recognition
test_embedding_r100 = get_embeddings_resnet100("my_images/testtom.jpg")
print(f"{test_embedding_r100}")
identity_r100, confidence_r100 = recognize_face(test_embedding_r100, face_db_r100, threshold=0.6)
print(f"[ResNet100] Identified as: {identity_r100} (Confidence: {confidence_r100:.2f})")


Face detected in my_images/testtom.jpg
tensor([ 3.2313e-01,  1.1005e-01,  1.1597e+00, -4.2847e-01,  1.2794e+00,
        -1.8125e+00,  1.9847e-01,  1.7196e+00, -2.1364e+00, -1.2774e+00,
        -2.7305e-02,  6.3181e-01,  7.3737e-04, -1.0055e+00, -5.5254e-01,
        -4.5089e-01, -3.7118e-01,  9.7012e-01, -7.1370e-01,  4.6684e-01,
         1.3359e-01, -6.8961e-02,  8.2523e-01, -3.5614e-01,  5.6834e-01,
        -7.9130e-01, -1.9675e-01,  8.6179e-02,  5.8092e-01, -1.8547e+00,
         6.0695e-01,  4.4940e-01,  5.9072e-01,  7.3975e-01, -2.5839e-01,
        -1.8900e-01, -3.8424e-01, -5.6531e-01, -9.3290e-02, -6.6885e-01,
        -3.6736e-01, -7.8731e-01, -4.6176e-01,  1.6864e-02, -3.1785e-01,
         9.8286e-01, -1.7032e-01, -5.4189e-01, -6.8212e-01, -4.9764e-01,
        -1.3406e-01, -1.7799e+00, -1.2717e+00, -5.4766e-01,  1.0330e+00,
        -3.3866e-01, -2.6705e+00,  2.0930e+00, -5.3282e-01, -4.0970e-01,
        -9.1128e-03, -3.5588e-02, -8.4441e-01,  5.2098e-02, -2.7658e-01,
        -6.0